# ARGCN 环境配置教程

## 概述

这份 notebook 是 ARGCN 教学系列的第 1 部分。这里的 **ARGCN** 指代本教程对 `reaction-gcnn` 中
`train_attention_model.py + models/relgcn.py + models/attention_readout.py` 这一条主线做出的
**PyTorch 教学复现版本**。

原仓库使用的是 **Chainer + chainer-chemistry**。为了适配当前更常见、也更易维护的环境，本教程不再要求读者安装
Chainer，而是使用 **conda + PyTorch + RDKit** 复现同样的数据流和模型结构。

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 教学版依赖设计：为什么从 Chainer 迁移到 Torch |
| 2 | 使用 conda 创建独立环境 |
| 3 | 注册 Jupyter kernel |
| 4 | 验证 Torch / RDKit / 数据文件路径 |
| 5 | 对照原仓库查看源码结构 |


## 1. 核心依赖一览

原仓库的时间背景是 2020 年前后，因此依赖栈明显偏旧。教学版不追求复现旧环境，而是复现**算法主线**：

| 模块 | 原仓库实现 | 教学版实现 |
|------|------------|------------|
| 深度学习框架 | Chainer | PyTorch |
| 分子处理 | RDKit | RDKit |
| 图网络实现 | chainer-chemistry | 纯 Torch 最小复现 |
| notebook 环境 | 未约束 | conda 独立环境 |

> 说明
>
> 为了适配“当前 channel 下的最新兼容版本”，下面的 conda 安装命令只固定 `python=3.11`，其余核心包交给 channel 解出当前最新兼容组合。
> 这样比硬编码一组很快过时的精确小版本更稳妥。


In [2]:
import os
from pathlib import Path

# ====== Step 1: 定位项目根目录与关键路径 ======
def find_project_root(start=None):
    here = Path(start or os.getcwd()).resolve()
    for path in [here, *here.parents]:
        if (path / 'teaching_demos').exists() and (path / 'source_repos').exists():
            return path
    raise RuntimeError('未找到项目根目录，请确认当前 notebook 位于 Chemical_Synthesis 项目内。')

PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / 'teaching_demos' / '4.reaction_condition_tutorial' / 'ARGCN'
REPO_DIR = PROJECT_ROOT / 'source_repos' / 'reaction-gcnn'
ENV_DIR = PROJECT_ROOT / 'envs' / 'argcn_tutorial_envs'
DEMO_DATA = TUTORIAL_DIR / 'data' / 'demo_data.csv'

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'TUTORIAL_DIR = {TUTORIAL_DIR}')
print(f'REPO_DIR = {REPO_DIR}')
print(f'ENV_DIR = {ENV_DIR}')
print(f'DEMO_DATA = {DEMO_DATA}')


PROJECT_ROOT = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
TUTORIAL_DIR = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/4.reaction_condition_tutorial/ARGCN
REPO_DIR = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/reaction-gcnn
ENV_DIR = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/argcn_tutorial_envs
DEMO_DATA = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/4.reaction_condition_tutorial/ARGCN/data/demo_data.csv


## 2. 使用 conda 创建教学环境

### 为什么这里必须使用 conda？

1. RDKit 在 conda-forge 上的安装最稳定。
2. PyTorch 可以通过 `pytorch` channel 直接拿到当前兼容版本。
3. 读者后续还会在 notebook 中切换 kernel，conda 比 `venv` 更适合这类科研工作流。

下面这段命令会在项目根目录下创建：`envs/argcn_tutorial_envs`


In [3]:
%%bash -s "$ENV_DIR"

ENV_DIR="$1"

if ! command -v conda >/dev/null 2>&1; then
  FOUND_CONDA_SH=""
  for candidate in "$HOME/miniconda3/etc/profile.d/conda.sh" "$HOME/anaconda3/etc/profile.d/conda.sh" "$HOME/software/miniconda3/etc/profile.d/conda.sh" "/opt/conda/etc/profile.d/conda.sh"; do
    if [ -f "$candidate" ]; then
      FOUND_CONDA_SH="$candidate"
      break
    fi
  done

  if [ -z "$FOUND_CONDA_SH" ]; then
    echo "未找到 conda，请先安装 Miniconda 或 Anaconda。"
    exit 1
  fi
  source "$FOUND_CONDA_SH"
else
  source "$(conda info --base)/etc/profile.d/conda.sh"
fi

if [ ! -x "$ENV_DIR/bin/python" ]; then
  conda create -y -p "$ENV_DIR" python=3.11
else
  echo "==> 环境已存在，跳过 conda create: $ENV_DIR"
fi

conda install -y -p "$ENV_DIR" -c pytorch -c conda-forge \
  pytorch cpuonly rdkit pandas numpy matplotlib scikit-learn jupyterlab ipykernel tqdm jupyter-collaboration

echo "==> conda 环境准备完成: $ENV_DIR"

==> 环境已存在，跳过 conda create: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_

AIDD/Chemical_Synthesis/envs/argcn_tutorial_envs


Channels:
 - pytorch
 - conda-forge
 - defaults
Platform: linux-64

data.json): ...working... 

done
Solving environment: ...working... 

...



==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version:

 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda





# All requested packages already installed.



==> conda 环境准备完成: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical

_Synthesis/envs/argcn_tutorial_envs


### 可选：如果你后续需要 GPU

教学 notebook 默认采用 CPU 方案，因为它最容易复现。如果你确实要在 GPU 上跑，可以把上面的 `cpuonly`
替换成 PyTorch 官方安装矩阵中当前推荐的 `pytorch-cuda` 组合。

教学逻辑本身与 CPU / GPU 无关。


## 3. 注册为 Jupyter Kernel（推荐）

这样做的目的是让后续 2 份 notebook 都可以直接切换到同一个教学环境。


In [4]:
%%bash -s "$ENV_DIR"

ENV_DIR="$1"
"$ENV_DIR/bin/python" -m ipykernel install --user --name argcn_tutorial --display-name "ARGCN Tutorial"
echo "==> Kernel 'ARGCN Tutorial' 已注册完成"

Installed kernelspec argcn_tutorial in /tmp/argcn_jupyter_data/kernels/argcn_tutorial


==> Kernel 'ARGCN Tutorial' 已注册完成


## 4. 验证环境

> 注意
>
> 在执行下面的验证代码之前，请先把 notebook kernel 切换到 `ARGCN Tutorial`。


In [5]:
# ====== Step 4.1: 验证核心依赖 ======
import sys
import numpy as np
import pandas as pd
import torch
import sklearn
from rdkit import rdBase

print(f'Python   : {sys.version.split()[0]}')
print(f'NumPy    : {np.__version__}')
print(f'pandas   : {pd.__version__}')
print(f'PyTorch  : {torch.__version__} (CUDA available: {torch.cuda.is_available()})')
print(f'sklearn  : {sklearn.__version__}')
print(f'RDKit    : {rdBase.rdkitVersion}')


Python   : 3.11.6
NumPy    : 2.4.3
pandas   : 3.0.1
PyTorch  : 2.5.1 (CUDA available: False)
sklearn  : 1.8.0
RDKit    : 2026.03.1


## 5. 验证 RDKit 与 Torch 的基础图操作

这个检查不是为了“训练模型”，而是为了确认后续两份 notebook 依赖的两个核心步骤都可用：

1. RDKit 能把 SMILES 变成分子对象。
2. Torch 能对邻接矩阵和节点特征做张量运算。


In [6]:
# ====== Step 5.1: RDKit 分子解析 + Torch 张量运算 ======
from rdkit import Chem

smiles = 'c1ccccc1Br'
mol = Chem.MolFromSmiles(smiles)
atom_ids = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
n = len(atom_ids)

adj = torch.zeros((n, n), dtype=torch.float32)
for bond in mol.GetBonds():
    i = bond.GetBeginAtomIdx()
    j = bond.GetEndAtomIdx()
    adj[i, j] = 1.0
    adj[j, i] = 1.0

x = torch.tensor(atom_ids, dtype=torch.float32).unsqueeze(-1)
aggregated = adj @ x

print('SMILES:', smiles)
print('atom_ids:', atom_ids)
print('adj shape:', tuple(adj.shape))
print('x shape:', tuple(x.shape))
print('adj @ x shape:', tuple(aggregated.shape))
print('aggregated[:5]:')
print(aggregated[:5])


SMILES: c1ccccc1Br
atom_ids: [6, 6, 6, 6, 6, 6, 35]
adj shape: (7, 7)
x shape: (7, 1)
adj @ x shape: (7, 1)
aggregated[:5]:
tensor([[12.],
        [12.],
        [12.],
        [12.],
        [12.]])


## 6. `reaction-gcnn` 源码结构概览

这一步不是装饰，而是后面两份教学 notebook 的地图。

### 本教程重点对应的源码文件

| 文件 | 作用 |
|------|------|
| `source_repos/reaction-gcnn/train.py` | 基础版三分子图拼接模型 |
| `source_repos/reaction-gcnn/train_attention_model.py` | 带 reaction-level attention 的版本 |
| `source_repos/reaction-gcnn/models/relgcn.py` | 关系图卷积编码器 |
| `source_repos/reaction-gcnn/models/attention_readout.py` | 三分子级别的注意力聚合 |
| `source_repos/reaction-gcnn/dataset/suzuki_data_frame_parser.py` | 数据解析与标签向量构造 |


In [7]:
# ====== Step 6.1: 打印关键源码树 ======
from pathlib import Path

key_paths = [
    REPO_DIR / 'train.py',
    REPO_DIR / 'train_attention_model.py',
    REPO_DIR / 'models' / 'relgcn.py',
    REPO_DIR / 'models' / 'attention_readout.py',
    REPO_DIR / 'models' / 'classifier.py',
    REPO_DIR / 'dataset' / 'suzuki_data_frame_parser.py',
    DEMO_DATA,
]

for path in key_paths:
    print(path.relative_to(PROJECT_ROOT), '->', path.exists())


source_repos/reaction-gcnn/train.py -> True
source_repos/reaction-gcnn/train_attention_model.py -> True
source_repos/reaction-gcnn/models/relgcn.py -> True
source_repos/reaction-gcnn/models/attention_readout.py -> True
source_repos/reaction-gcnn/models/classifier.py -> True
source_repos/reaction-gcnn/dataset/suzuki_data_frame_parser.py -> True
teaching_demos/4.reaction_condition_tutorial/ARGCN/data/demo_data.csv -> True


## 7. 快速理解 ARGCN 教学版的工作流

```text
demo_data.csv
    ↓
反应 SMILES 拆分为 Reactant1 / Reactant2 / Product
    ↓
条件名称映射为全局多热标签向量
    ↓
每个分子转成 (atom_ids, relation_adjs)
    ↓
共享 RelGCN 编码器
    ↓
reaction-level attention readout
    ↓
多标签条件预测 logits
```

这也是后面两份 notebook 的顺序：

- `2_数据处理.ipynb`: 负责把原始文本变成图张量和标签向量
- `3_模型展示.ipynb`: 负责把这些张量送入 Torch 版 ARGCN，演示推理路径


## 8. 总结

这一份 notebook 完成了三件事：

1. 用 **conda** 明确了教学环境的构建方式。
2. 把原始 Chainer 工程替换成了更适合当前学习和维护的 **Torch 教学环境**。
3. 给后续两份 notebook 标出了准确的源码映射路径。

下一步建议直接打开 `2_数据处理.ipynb`，先把 `reaction-gcnn` 的三分子输入和多标签条件向量看懂。
